# Modelagem para o problema X-Health

## Importando Bibliotecas

In [ ]:
#adicionando retorno no diretório ao caminho
import sys
sys.path.append('../')

#ignorar warnings 
import warnings
warnings.filterwarnings('ignore')

#informação de diretórios
from src.globals import *
from src.models import *
from src.auxiliar_xhealth import *

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from xgboost import XGBClassifier
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Adicionando base e definindo var_alvo

In [ ]:
#importando a base do arquivo externo
nome_base = f'{EXTERNAL_DATA_DIR}/dataset_2021-5-26-10-14.csv'
df = pd.read_csv(nome_base, sep = '\t', encoding='utf-8', na_values="missing")
logger.info(f'Base importada com tamanho: {len(df)}')

In [4]:
#Definir a variavel alvo
var_alvo = 'default'  # Variável alvo

In [ ]:
df.columns.values

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [9]:
df_tratado = tratar_categoricas(df)

In [ ]:
df_tratado.isna().sum().sort_values(ascending=False)

# Transformações e normalizações

In [12]:
## preparar variaveis e separar em teste e treino
X_train, X_test, y_train, y_test = preparar_dados(df_tratado, target='default')

In [ ]:
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt

def selecionar_pelo_xgboost(X_train, y_train, top_n=10):
    """
    Treina um modelo XGBoost e seleciona as principais variáveis com base na importância.

    Parâmetros:
    -----------
    X_train : pd.DataFrame
        Conjunto de treino contendo apenas as variáveis preditoras.
    y_train : pd.Series
        Variável alvo (default = 0 ou 1).
    top_n : int, opcional (default=10)
        Número de variáveis mais importantes a serem selecionadas.

    Retorno:
    --------
    list
        Lista com as variáveis mais importantes.
    """
    
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train)
    
    # Importância das features
    importance = pd.DataFrame({"Variável": X_train.columns, "Importância": model.feature_importances_})
    importance = importance.sort_values(by="Importância", ascending=False).head(top_n)
    
    # Plot da importância das variáveis
    plt.figure(figsize=(10,6))
    plt.barh(importance["Variável"], importance["Importância"], color="blue")
    plt.xlabel("Importância")
    plt.ylabel("Variáveis")
    plt.title("Importância das Variáveis - XGBoost")
    plt.gca().invert_yaxis()
    plt.show()
    
    return importance["Variável"].tolist()

# Exemplo de uso
top_features = selecionar_pelo_xgboost(X_train, y_train, top_n=15)
print("Variáveis Selecionadas:", top_features)

In [13]:
# Criando os DMatrix para XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [14]:
# Configuração do modelo
params = {
    'objective': 'binary:logistic',  # Classificação binária
    'eval_metric': 'auc',        # Log Loss como métrica de erro
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100
}

### Modelo pre-optuna

In [15]:
# Treinando o modelo
model = xgb.train(params, dtrain, num_boost_round=100)

In [ ]:
# Chamando a função de avaliação:
metrica_inicial = avaliar_XGBoost(model, dtrain, y_train, dtest, y_test)
metrica_inicial

In [17]:
# Previsões de probabilidades
y_pred_train_prob = model.predict(dtrain)
y_pred_test_prob = model.predict(dtest)

# Convertendo as probabilidades para classificação binária (0 ou 1)
y_pred_train = (y_pred_train_prob > 0.5).astype(int)
y_pred_test = (y_pred_test_prob > 0.5).astype(int)

# Valores reais (y_true)
y_true_train = y_train.values
y_true_test = y_test.values

In [ ]:
plot_matriz_confusao(y_true_train, y_pred_train, y_true_test, y_pred_test, 
                     size=(6,3), cmap_train="Reds", cmap_test="Blues", 
                     save_path=f'{FIGURES_DIR}/matriz_confusao_XGBoost_inicial.png')

#### Optuna

In [19]:
# desabilita warnings do optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Configurar o nível de log no Optuna
logger.disable("optuna")  # Desativa logs do Optuna no loguru

In [20]:
#metrica da função
metrica = "test-auc-mean"

# Função de otimização do Optuna
def objective(trial):
    
    """
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",  # Avalia a métrica AUC
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": 0.80,  # Mantendo fixo
        "colsample_bytree": 0.80,  # Mantendo fixo
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0),  # Regularização L2
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0),  # Regularização L1
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "n_jobs": -1  # Habilita paralelização
    }
    
    
    """    
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",  # Avalia a métrica AUC
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.79, 0.81),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.79, 0.81),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0), # Regularização L2
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0), # Regularização L1
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "n_jobs": -1  # Habilita paralelização
    }


    # Executa validação cruzada com 5 folds
    cv_results = xgb.cv(
        params, dtrain, num_boost_round=100,
        nfold=5, stratified=True, early_stopping_rounds=10, seed=42
    )

    # Exibir colunas disponíveis para depuração de metrica
    # print("Colunas disponíveis no cv_results:", cv_results.columns)

    # Capturar a métrica correta
    metric_column = [col for col in cv_results.columns if metrica in col]

    if not metric_column:
        raise ValueError(f"Métrica '{metrica}' não encontrada! Colunas disponíveis: {cv_results.columns}")
    return max(cv_results[metric_column[0]])  # Maximiza AUC

In [21]:
# Executar a otimização com paralelização
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=20, n_jobs=-1)  # Executa múltiplos trials em paralelo

In [ ]:
# Melhor conjunto de hiperparâmetros encontrado
best_params = study.best_params
best_value = study.best_value
print("Melhores Hiperparâmetros:", best_params)
print("Melhor Valor:", best_value)

#Melhores Hiperparâmetros: {'max_depth': 9, 'learning_rate': 0.29270414309112364, 'n_estimators': 492, 'subsample': 0.7963351488280979, 'colsample_bytree': 0.8052392449589032, 'lambda': 9.14174843111581, 'alpha': 4.4233718970157465, 'min_child_weight': 4, 'gamma': 0.03469453368455744}
#Melhor Valor: 0.9242312402850119

### Modelo pós-optuna

In [23]:
# Treinar modelo final com melhores hiperparâmetros
xgb_optimized = xgb.train(best_params, dtrain, num_boost_round=100)# Treinar modelo final com melhores hiperparâmetros

In [24]:
# Chamando a função de avaliação:
metrica_final = avaliar_XGBoost(xgb_optimized, dtrain, y_train, dtest, y_test)

In [ ]:
metrica_comparada = pd.merge(metrica_inicial, metrica_final, on="Métrica")
metrica_comparada

In [ ]:
plot_matriz_confusao(y_true_train, y_pred_train, y_true_test, y_pred_test, 
                     size=(6,3), cmap_train="Reds", cmap_test="Blues", 
                     save_path=f'{FIGURES_DIR}/matriz_confusao_XGBoost_final.png')